# Random Forest — 5-Seed Stability Check

## Purpose
Run the Random Forest baseline with the **same fixed observation-level train/test split** used for the Standard QML and PAQFE-v2 seed experiments.

Only the Random Forest's internal random seed changes: 42–46. The data split remains fixed at `test_size=0.20, random_state=42`.

Model: `RandomForestRegressor(n_estimators=300, random_state=seed, n_jobs=-1)`.

This gives a directly comparable 5-seed stability table while also reporting training-vs-test performance to examine the generalization gap.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEEDS = [42, 43, 44, 45, 46]
TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
N_ESTIMATORS = 300

FEATURES = [
    'xAl','VEC','dHmix','dSmix','delta',
    'Omega','dchi','Mean_Tm','Temperature_C','log10_Strain_rate'
]
TARGET = 'UTS_MPa'
COMPOSITION_COLS = ['Al_norm','Co_norm','Cr_norm','Fe_norm','Ni_norm']


## 1. Load the same Excel workbook (`B_Model_Ready`)

In [ ]:
from google.colab import files
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
df = pd.read_excel(file_path, sheet_name='B_Model_Ready')
required = FEATURES + [TARGET] + COMPOSITION_COLS
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
df[required] = df[required].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=required).copy()
df['Composition_Group'] = df[COMPOSITION_COLS].round(3).astype(str).agg('_'.join, axis=1)
print('Dataset shape:', df.shape)
print('Observations:', len(df))
print('Unique compositions:', df['Composition_Group'].nunique())


## 2. Fixed observation-level split
This is the same `80/20` split protocol used for the QML seed comparison. It is created **once** and reused for every RF seed.

In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE
)

print('Training observations:', len(X_train))
print('Testing observations :', len(X_test))


In [ ]:
train_results = []
feature_importance_records = []

for seed in SEEDS:
    rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=seed,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    train_pred = rf.predict(X_train)
    test_pred = rf.predict(X_test)

    train_mae = mean_absolute_error(y_train, train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_r2 = r2_score(y_train, train_pred)

    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    train_results.append({
        'Seed': seed,
        'Train_MAE': train_mae,
        'Train_RMSE': train_rmse,
        'Train_R2': train_r2,
        'Test_MAE': test_mae,
        'Test_RMSE': test_rmse,
        'Test_R2': test_r2,
        'MAE_Gap': test_mae - train_mae,
        'RMSE_Gap': test_rmse - train_rmse,
        'R2_Gap': train_r2 - test_r2
    })

    for f, imp in zip(FEATURES, rf.feature_importances_):
        feature_importance_records.append({
            'Seed': seed,
            'Feature': f,
            'Importance': imp
        })

    print(
        f'Seed {seed}: '
        f'Test MAE={test_mae:.3f}, '
        f'Test RMSE={test_rmse:.3f}, '
        f'Test R2={test_r2:.3f}'
    )


In [ ]:
rf_seed_results = pd.DataFrame(train_results)

print('===== Random Forest — 5 Seed Results =====')
display(rf_seed_results)

print('===== Test Mean ± SD =====')
rf_summary = pd.DataFrame({
    'Metric': ['MAE','RMSE','R2'],
    'Mean': [
        rf_seed_results['Test_MAE'].mean(),
        rf_seed_results['Test_RMSE'].mean(),
        rf_seed_results['Test_R2'].mean()
    ],
    'SD': [
        rf_seed_results['Test_MAE'].std(ddof=1),
        rf_seed_results['Test_RMSE'].std(ddof=1),
        rf_seed_results['Test_R2'].std(ddof=1)
    ]
})
display(rf_summary)


## 3. Average feature importance
This is optional but useful for explaining why RF performs strongly with a small dataset. It is descriptive only; it does not prove causal/material importance.

In [ ]:
importance_df = pd.DataFrame(feature_importance_records)
importance_summary = (
    importance_df.groupby('Feature')['Importance']
    .agg(['mean','std'])
    .sort_values('mean', ascending=False)
    .reset_index()
)
display(importance_summary)
importance_summary.to_csv('RF_feature_importance_5_seed.csv', index=False)


## 4. Save results
Use the 5-seed test mean ± SD for direct comparison with the Standard QML and PAQFE-v2 seed experiments. Keep the train/test gaps for overfitting discussion.

In [ ]:
rf_seed_results.to_csv('RF_5_seed_observation_results.csv', index=False)
rf_summary.to_csv('RF_5_seed_observation_summary.csv', index=False)

print('Saved:')
print('- RF_5_seed_observation_results.csv')
print('- RF_5_seed_observation_summary.csv')
print('- RF_feature_importance_5_seed.csv')


## Interpretation checkpoint

Compare the RF test mean ± SD with:

Standard QML: **MAE 217.43 ± 5.61, RMSE 272.41 ± 7.45, R² 0.232 ± 0.042**.

PAQFE-v2 (slow alpha): **MAE 212.01 ± 16.31, RMSE 266.48 ± 22.61, R² 0.262 ± 0.126**.

Do not conclude that a model is generally superior from this observation-level split alone. Use the composition-level validation for the main generalization claim.